# Jacobian lens — walkthrough

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation.

In [1]:
!git clone https://github.com/anthropics/jacobian-lens.git
%cd jacobian-lens

Cloning into 'jacobian-lens'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 67 (delta 0), reused 0 (delta 0), pack-reused 65 (from 2)
Receiving objects: 100% (67/67), 1.90 MiB | 8.73 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/jacobian-lens


In [2]:
!pip install -e .

Obtaining file:///content/jacobian-lens
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jlens (pyproject.toml) ... done
  Created wheel for jlens: filename=jlens-0.1.0-0.editable-py3-none-any.whl size=8901 sha256=d33c9bb7caa2c0501f4bd01a12c24b71374821bf7aa52eb9cb6b3338c2709545
  Stored in directory: /tmp/pip-ephem-wheel-cache-jmf3smxr/wheels/9b/16/f6/ff5117e12d375117559a1ded186e4d458b172d145efd7f032b
Successfully built jlens


In [3]:
import jlens

jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

## 1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface

In [4]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

HFLensModel(Qwen3_5ForCausalLM, n_layers=32, d_model=2560)

## 2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [5]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))

## 3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports each layer's residual into the final-layer basis with `J_l`, and decodes through the model's own unembedding. `use_jacobian=False` skips the transport — that's the vanilla logit lens.

Below: a two-hop factual question, read out at the boot token. The J-lens surfaces interpretable tokens at layers where the logit lens is still noise.

In [12]:
prompt = "Fact: The currency used in the country shaped like a boot is"
tokens = tokenizer(prompt, return_tensors="pt")["input_ids"][0]

print("Number of tokens:", len(tokens))
print([tokenizer.decode([t]) for t in tokens])

Number of tokens: 13
['Fact', ':', ' The', ' currency', ' used', ' in', ' the', ' country', ' shaped', ' like', ' a', ' boot', ' is']


In [6]:
prompt = "Fact: The currency used in the country shaped like a boot is"
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-2])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-2], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

L  8 logit-lens: ['oman', 'edom', 'ולי', ' Urlaubs', 'GPC']
L  8 J-lens:     [' `', ' boots', ' *', ' `\\', ' heel']
L 16 logit-lens: ['shaw', 'วย', 'amaz', 'REA', '举世']
L 16 J-lens:     ['?', '？', "'?", '____', ' Italy']
L 24 logit-lens: ['的形状', '形状的', '形状', 'shape', '-shaped']
L 24 J-lens:     ['-shaped', ' shape', ' shaped', 'shape', '形状']
L 30 logit-lens: [' is', ' shape', '-shaped', ' heel', ' shaped']
L 30 J-lens:     [' is', ' shape', ' shaped', '-shaped', ' heel']
model:           [' is', ' in', ' on', '.', ' with']


In [17]:
import gc
import torch
import pandas as pd

def run_prompt(prompt, model, tokenizer, lens):
    """
    Extract top-1 lens token for every:
        layer × source position

    Uses only one layer and one position per lens.apply() call
    to minimize peak GPU memory.
    """

    input_ids = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"][0]

    seq_len = len(input_ids)

    # Every source position in the prompt
    positions = list(range(seq_len))

    # Only layers for which this lens was fitted
    layers = list(lens.source_layers)

    rows = []

    for layer in layers:
        print(f"Layer {layer}/{layers[-1]}")

        for pos in positions:

            try:
                lens_logits, model_logits, returned_ids = lens.apply(
                    model,
                    prompt,
                    layers=[layer],
                    positions=[pos],
                )

                # Shape should be [1, vocab_size]
                logits = lens_logits[layer][0]

                # top-1 token
                top1_id = int(torch.argmax(logits).item())
                top1_token = tokenizer.decode([top1_id])

                # Source token at this position
                source_id = int(input_ids[pos].item())
                source_token = tokenizer.decode([source_id])

                rows.append({
                    "layer": layer,
                    "position": pos,
                    "source_token": source_token,
                    "top1_token_id": top1_id,
                    "top1_token": top1_token,
                })

            except torch.OutOfMemoryError:
                print(
                    f"OOM at layer={layer}, position={pos}. "
                    "Skipping this cell."
                )
                rows.append({
                    "layer": layer,
                    "position": pos,
                    "source_token": tokenizer.decode(
                        [int(input_ids[pos].item())]
                    ),
                    "top1_token_id": None,
                    "top1_token": None,
                })

            finally:
                # Release every large GPU tensor immediately
                for name in [
                    "lens_logits",
                    "model_logits",
                    "returned_ids",
                    "logits",
                ]:
                    if name in locals():
                        del locals()[name]

                gc.collect()
                torch.cuda.empty_cache()

    return pd.DataFrame(rows)

In [18]:
prompt = "Fact: The currency used in the country shaped like a boot is"

df = run_prompt(
    prompt,
    model,
    tokenizer,
    lens,
)

df.head()

Layer 0/30
OOM at layer=0, position=0. Skipping this cell.
Layer 1/30
Layer 2/30
Layer 3/30
Layer 4/30
Layer 5/30
Layer 6/30
Layer 7/30
Layer 8/30
Layer 9/30
Layer 10/30
Layer 11/30
Layer 12/30
Layer 13/30
Layer 14/30
Layer 15/30
Layer 16/30
Layer 17/30
Layer 18/30
Layer 19/30
Layer 20/30
Layer 21/30
Layer 22/30
Layer 23/30
Layer 24/30
Layer 25/30
Layer 26/30
Layer 27/30
Layer 28/30
Layer 29/30
Layer 30/30


,layer,position,source_token,top1_token_id,top1_token
0,0,0,Fact,NaN,None
1,0,1,:,26.0,;
2,0,2,The,208731.0,、
3,0,3,currency,63558.0,(...)
4,0,4,used,328.0,""""


In [19]:
grid = df.pivot(
    index="layer",
    columns="position",
    values="top1_token",
)

grid

position,0,1,2,3,4,5,6,7,8,9,10,11,12
layer,,,,,,,,,,,,,
0,None,;,、,(...),"""",""".",",",--,``,``,"""",--,'
1,``,:,``,[...],...,###,",",--,``,``,``,``,...
2,اً,\n\n,<|endoftext|>,�,to,",",",",""",",\n,a,s,`,","
3,,...,The,<|im_end|>,...,""".",",",。。,\n,\n\n\n,_,--,...
4,,___,‘,<|im_end|>,...,""".",.,。。,…,\n\n\n,.,`,.
5,,...,‘,[...],...,""".",.,__,\n,__,.,`,...
6,<|endoftext|>,,‘,currency,\n,""".",\n\n,\\,\n,__,_,`,...
7,<|endoftext|>,,__,currency,[...],""".","\""",__,\n,__,_,`,...
8,\n,,__,currency,\n,currency,___,__,\n,__,currency,`,...


In [29]:
prompts = [
    "My sister has always wanted to visit France",
    "The cheese they served reminded him of India",
    "My aunt has always wanted to visit Germany",
]

all_results = []

for prompt_id, prompt in enumerate(prompts):
    print(f"\n=== Prompt {prompt_id} ===")

    df_prompt = run_prompt(
        prompt,
        model,
        tokenizer,
        lens,
    )

    df_prompt["prompt_id"] = prompt_id
    df_prompt["prompt"] = prompt

    all_results.append(df_prompt)

results = pd.concat(all_results, ignore_index=True)

results.head()


=== Prompt 0 ===
Layer 0/30
Layer 1/30
Layer 2/30
Layer 3/30
Layer 4/30
Layer 5/30
Layer 6/30
Layer 7/30
Layer 8/30
Layer 9/30
Layer 10/30
Layer 11/30
Layer 12/30
Layer 13/30
Layer 14/30
Layer 15/30
Layer 16/30
Layer 17/30
Layer 18/30
Layer 19/30
Layer 20/30
Layer 21/30
Layer 22/30
Layer 23/30
Layer 24/30
Layer 25/30
Layer 26/30
Layer 27/30
Layer 28/30
Layer 29/30
Layer 30/30

=== Prompt 1 ===
Layer 0/30
Layer 1/30
Layer 2/30
Layer 3/30
Layer 4/30
Layer 5/30
Layer 6/30
Layer 7/30
Layer 8/30
Layer 9/30
Layer 10/30
Layer 11/30
Layer 12/30
Layer 13/30
Layer 14/30
Layer 15/30
Layer 16/30
Layer 17/30
Layer 18/30
Layer 19/30
Layer 20/30
Layer 21/30
Layer 22/30
Layer 23/30
Layer 24/30
Layer 25/30
Layer 26/30
Layer 27/30
Layer 28/30
Layer 29/30
Layer 30/30

=== Prompt 2 ===
Layer 0/30
Layer 1/30
Layer 2/30
Layer 3/30
Layer 4/30
Layer 5/30
Layer 6/30
Layer 7/30
Layer 8/30
Layer 9/30
Layer 10/30
Layer 11/30
Layer 12/30
Layer 13/30
Layer 14/30
Layer 15/30
Layer 16/30
Layer 17/30
Layer 18/30
Laye

,layer,position,source_token,top1_token_id,top1_token,prompt_id,prompt
0,0,0,My,696,_,0,My sister has always wanted to visit France
1,0,1,sister,271,\n\n,0,My sister has always wanted to visit France
2,0,2,has,361,<,0,My sister has always wanted to visit France
3,0,3,always,1137,--,0,My sister has always wanted to visit France
4,0,4,wanted,9609,``,0,My sister has always wanted to visit France


## 4. Render a slice page (inline)

`compute_slice` + `build_page` produce an interactive position × layer view of the lens's token ranks (the `?` in the corner explains the controls). `mode="embed"` inlines everything so the page is self-contained.

In [30]:
prompt = "My sister has always wanted to visit France"

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    mask_display=True,
)

page, _, _ = build_page(
    slice_data,
    prompt,
    title="Custom prompt",
    description=prompt,
    alt_token=gloss,
)

notebook_iframe(page)

In [31]:
prompt = "The cheese they served reminded him of India"

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    mask_display=True,
)

page, _, _ = build_page(
    slice_data,
    prompt,
    title="Custom prompt",
    description=prompt,
    alt_token=gloss,
)

notebook_iframe(page)

In [32]:
prompt = "My aunt has always wanted to visit Germany"

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    mask_display=True,
)

page, _, _ = build_page(
    slice_data,
    prompt,
    title="Custom prompt",
    description=prompt,
    alt_token=gloss,
)

notebook_iframe(page)

In [7]:
import gzip
import json

from jlens.examples import EXAMPLES, resolve_prompt
from jlens.vis import build_page, compute_slice, notebook_iframe

# English gloss for Qwen's Chinese/Japanese/Korean vocab tokens (machine-
# generated, best-effort), shown next to
# the token in the page (alt_token=).
gloss = {
    int(k): v for k, v in json.load(gzip.open("assets/qwen_gloss.json.gz")).items()
}

example = next(e for e in EXAMPLES if e.slug == "multihop")
prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
)
page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
)
notebook_iframe(page)